### Conda Environment Check

In [2]:
from __future__ import print_function
from packaging.version import parse as Version
from platform import python_version

OK = '\x1b[42m[ OK ]\x1b[0m'
FAIL = "\x1b[41m[FAIL]\x1b[0m"

try:
    import importlib
except ImportError:
    print(FAIL, "Python version 3.12.10 is required,"
                " but %s is installed." % sys.version)

def import_version(pkg, min_ver, fail_msg=""):
    mod = None
    try:
        mod = importlib.import_module(pkg)
        if pkg in {'PIL'}:
            ver = mod.VERSION
        else:
            ver = mod.__version__
        if Version(ver) == Version(min_ver):
            print(OK, "%s version %s is installed."
                  % (lib, min_ver))
        else:
            print(FAIL, "%s version %s is required, but %s installed."
                  % (lib, min_ver, ver))    
    except ImportError:
        print(FAIL, '%s not installed. %s' % (pkg, fail_msg))
    return mod


# first check the python version
pyversion = Version(python_version())

if pyversion >= Version("3.12.10"):
    print(OK, "Python version is %s" % pyversion)
elif pyversion < Version("3.12.10"):
    print(FAIL, "Python version 3.12.10 is required,"
                " but %s is installed." % pyversion)
else:
    print(FAIL, "Unknown Python version: %s" % pyversion)

    
print()
requirements = {'numpy': "2.2.5", 'matplotlib': "3.10.1",'sklearn': "1.6.1", 
                'pandas': "2.2.3",'xgboost': "3.0.0", 'shap': "0.47.2", 
                'polars': "1.27.1", 'seaborn': "0.13.2"}

# now the dependencies
for lib, required_version in list(requirements.items()):
    import_version(lib, required_version)

[ OK ] Python version is 3.12.10

[ OK ] numpy version 2.2.5 is installed.
[ OK ] matplotlib version 3.10.1 is installed.
[ OK ] sklearn version 1.6.1 is installed.
[ OK ] pandas version 2.2.3 is installed.
[ OK ] xgboost version 3.0.0 is installed.
[ OK ] shap version 0.47.2 is installed.
[ OK ] polars version 1.27.1 is installed.
[ OK ] seaborn version 0.13.2 is installed.


### Load the Merged Application Dataset

In [3]:
import pandas as pd
import matplotlib
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_rows', 150)


In [4]:
# read in the dataset
df_full_application = pd.read_csv('../data/application_merged_data.csv')
df_full = df_full_application.copy()
# First view of the dataset
print("Example of the application_merged_data dataset:")
print(df_full.head())
print("\n Shape of the application_merged_data dataset:")
print(df_full.shape)

Example of the application_merged_data dataset:
   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  \
0      100002       1         Cash loans           M            N               Y             0          202500.0   
1      100003       0         Cash loans           F            N               N             0          270000.0   
2      100004       0    Revolving loans           M            Y               Y             0           67500.0   
3      100006       0         Cash loans           F            N               Y             0          135000.0   
4      100007       0         Cash loans           M            N               Y             0          121500.0   

   AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE NAME_TYPE_SUITE NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0    406597.5      24700.5         351000.0   Unaccompanied          Working  Secondary / secondary special   
1   1293502.5      35698.5 

##### Columns in the Merged Application Dataset

In [5]:
print("Basic Information of the application_merged_data dataset:")
print(df_full.info())
print("\n\nColumn Names:")
print(df_full.columns.tolist())

Basic Information of the application_merged_data dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 137 entries, SK_ID_CURR to PREV_STATUS_UNUSED OFFER
dtypes: float64(80), int64(41), object(16)
memory usage: 321.4+ MB
None


Column Names:
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIV

### Unpack feature metadata (categorizations)

In [6]:
import pickle
import pandas as pd
import numpy as np

# ==========================================
# Load Feature Metadata
# ==========================================
print("Loading feature metadata...")

with open('../data/feature_metadata.pkl', 'rb') as f:
    meta = pickle.load(f)

# ==========================================
# Unpack Variables
# ==========================================

all_features = meta['all_features']

cts_features = meta['cts_features']
cts_features_log = meta['cts_features_log']
cts_features_other = meta['cts_features_other']

cat_features = meta['cat_features']

ord_features_num = meta['ord_features_num']
ord_features_str = meta['ord_features_str']
ord_features = meta['ord_features']
education_order = meta['education_order']

binary_features = meta['binary_features']
binary_features_num = meta['binary_features_num']
binary_features_str = meta['binary_features_str']


# ==========================================
# Verification
# ==========================================
print("✅ Feature metadata loaded successfully!")
print(f"\nTotal Features (all_features):       {len(all_features)}")
print(f"\nContinuous Total (cts_features):     {len(cts_features)}")
print(f"Continuous (Log):                      {len(cts_features_log)}")
print(f"Continuous (Non-Log):                  {len(cts_features_other)}")
print(f"\nCategorical Total (cat_features):    {len(cat_features)}")
print(f"\nOrdinal Total (ord_features):        {len(ord_features)}")
print(f"Ordinal (num):                         {len(ord_features_num)}")
print(f"Ordinal (str):                         {len(ord_features_str)}")
print(f"Education Order check:                 {education_order}")
print(f"\nBinary Total:                        {len(binary_features)}")
print(f"Binary (num):                          {len(binary_features_num)}")
print(f"Binary (str):                          {len(binary_features_str)}")



Loading feature metadata...
✅ Feature metadata loaded successfully!

Total Features (all_features):       134

Continuous Total (cts_features):     117
Continuous (Log):                      10
Continuous (Non-Log):                  107

Categorical Total (cat_features):    14

Ordinal Total (ord_features):        3
Ordinal (num):                         2
Ordinal (str):                         1
Education Order check:                 ['Lower secondary', 'Secondary / secondary special', 'Incomplete higher', 'Higher education', 'Academic degree']

Binary Total:                        37
Binary (num):                          33
Binary (str):                          4


### Feature VS Target Variables

In [7]:
from sklearn.model_selection import train_test_split

X = df_full.drop(columns=['TARGET', 'SK_ID_CURR', 'CODE_GENDER'])

y = df_full['TARGET']

print(f"Feature Variables' shape: {X.shape}")
print(f"Target Variable's shape: {y.shape}")

Feature Variables' shape: (307511, 134)
Target Variable's shape: (307511,)


### Data Preprocessing

In [8]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder,FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, BayesianRidge
from sklearn.ensemble import RandomForestClassifier, ExtraTreesRegressor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# ==========================================
# 1. Preprocessors
# ==========================================
print("Preparing Preprocessors...")

# Ordinal Encoder
ord_encoder_unified = OrdinalEncoder(
    categories=[education_order], 
    handle_unknown='use_encoded_value', 
    unknown_value=np.nan 
)
# ord_encoder_linear = OrdinalEncoder(categories=[education_order], 
#                                     handle_unknown='use_encoded_value', unknown_value=np.nan)
# ord_encoder_tree = OrdinalEncoder(categories=[education_order], 
#                                   handle_unknown='use_encoded_value', unknown_value=-1)

xt_estimator = ExtraTreesRegressor(
    n_estimators=1, 
    max_depth=10, 
    min_samples_leaf=20,
    n_jobs=-1, 
    random_state=42
)

# Iterative Imputer
rf_imputer_estimator = RandomForestRegressor(
    n_estimators=1, 
    max_depth=10, 
    min_samples_leaf=20,
    max_samples=0.5,
    max_features='sqrt',
    random_state=42, 
    n_jobs=1
)
# max_iter=10
iterative_imp = IterativeImputer(
    estimator=rf_imputer_estimator,
    # estimator=xt_estimator,  
    max_iter=10, 
    random_state=42
)

# Bayesian Iterative Imputer
# iterative_imp = IterativeImputer(estimator=BayesianRidge(), max_iter=5, random_state=42)

# Log Transformer
log_transformer = FunctionTransformer(np.log1p, validate=False)



Preparing Preprocessors...


#### Regression Model Preprocessor (Impute NaN)

In [10]:
linear_preprocessor = ColumnTransformer(
    transformers=[
        # 1. Nominal Categorical (include binary_str): OneHot
        ('nominal', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features),
        
        # 2. Skewed Continuous (Log): Impute -> Log -> Scale
        ('skewed_cts', Pipeline([
            ('imputer', iterative_imp),
            ('log', log_transformer),   
            ('scaler', StandardScaler())
        ]), cts_features_log),
        
        # 3. Normal Continuous (Non-Log, 包含 binary_num): Impute -> Scale
        ('normal_cts', Pipeline([
            ('imputer', iterative_imp),
            ('scaler', StandardScaler())
        ]), cts_features_other),
        
        # 4. Ordinal String (Education): Encode -> Impute -> Scale
        ('ord_str', Pipeline([
            ('encoder', ord_encoder_unified),
            ('imputer', iterative_imp),
            ('scaler', StandardScaler())
        ]), ord_features_str),
        
        # 5. Ordinal Num (Rating): Impute -> Scale
        ('ord_num', Pipeline([
            ('imputer', iterative_imp),
            ('scaler', StandardScaler())
        ]), ord_features_num)
    ],
    verbose_feature_names_out=False
)

#### Tree Model Preprocessor (Native NaN)

In [ ]:
tree_preprocessor = ColumnTransformer(
    transformers=[
        # 1. Nominal Categorical: OneHot
        ('nominal', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features),
        
        # 2. Ordinal String (Education): Encode (NaN retained) -> Passthrough
        ('ord_str', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')), 
            ('encoder', ord_encoder_unified) 
        ]), ord_features_str),
        
        # 3. All numerical (Log/Non-Log/Ord_Num/Binary_Num)
        ('num_all', 'passthrough', cts_features_log + cts_features_other + ord_features_num)
    ],
    verbose_feature_names_out=False
)

### preprocessed data (Sample)

In [11]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split


data_save_dir = '../data/'
if not os.path.exists(data_save_dir):
    os.makedirs(data_save_dir)

print(">>> Generating Linear Processed Dataset...")

seed = 42 

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=seed
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=seed
)

print(f"Data Split -> Train: {X_train.shape}")

print("Fitting the linear_preprocessor...")


linear_preprocessor.fit(X_train, y_train)

print("Transforming Train set...")
X_train_processed = linear_preprocessor.transform(X_train)

print("Transforming Val set...")
X_val_processed = linear_preprocessor.transform(X_val)

# ==========================================
# 3. feature name
# ==========================================
try:
    feature_names = linear_preprocessor.get_feature_names_out()
except AttributeError:
    print("Warning: Could not extract feature names, using generic names.")
    feature_names = [f"feat_{i}" for i in range(X_train_processed.shape[1])]

# ==========================================
# 4. CSV
# ==========================================
df_train_proc = pd.DataFrame(X_train_processed, columns=feature_names)
df_val_proc = pd.DataFrame(X_val_processed, columns=feature_names)

df_train_proc['TARGET'] = y_train.values
df_val_proc['TARGET'] = y_val.values

train_path = os.path.join(data_save_dir, 'linear_processed_train_sample.csv')
test_path = os.path.join(data_save_dir, 'linear_processed_val_sample.csv')

df_train_proc.to_csv(train_path, index=False)
df_val_proc.to_csv(test_path, index=False)

print(f"\n Success! Files saved:")
print(f"   1. {train_path} (Rows: {len(df_train_proc)})")
print(f"   2. {test_path} (Rows: {len(df_val_proc)})")

>>> Generating Linear Processed Dataset...
Data Split -> Train: (184506, 134)
Fitting the linear_preprocessor...
Transforming Train set...
Transforming Val set...

 Success! Files saved:
   1. ../data/linear_processed_train_sample.csv (Rows: 184506)
   2. ../data/linear_processed_val_sample.csv (Rows: 61502)


In [31]:
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_validate

global_ratio = float(np.sum(y == 0)) / np.sum(y == 1)
print(f"Calculated Imbalance Ratio: {global_ratio:.2f}")

Calculated Imbalance Ratio: 11.39


In [ ]:
model_configs = {
    'Logistic Regression': (
        Pipeline([
            ('prep', linear_preprocessor),
            ('clf', LogisticRegression(class_weight='balanced', solver='saga', max_iter=2000, n_jobs=-1))
        ]),
        {'clf__C': [0.01, 0.1, 1], 'clf__penalty': ['l1', 'l2']}
    ),
    'Decision Tree': (
        Pipeline([
            ('prep', tree_preprocessor),
            ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=42))
        ]),
        {'clf__max_depth': [10, 15, 20], 'clf__min_samples_leaf': [20, 50, 100]}
    ),
    'Random Forest': (
        Pipeline([
            ('prep', tree_preprocessor),
            ('clf', RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42))
        ]),
        {'clf__n_estimators': [200, 300], 'clf__max_depth': [15, 20, 25, None], 'clf__max_features': ['sqrt', 0.5], 'clf__min_samples_leaf': [2, 5, 10]}
    ),
    'XGBoost': (
        Pipeline([
            ('prep', tree_preprocessor),
            ('clf', XGBClassifier(scale_pos_weight=global_ratio, tree_method='hist', eval_metric='aucpr', n_jobs=-1, random_state=42))
        ]),
        {'clf__n_estimators': [300, 500, 800], 'clf__learning_rate': [0.01, 0.05, 0.1], 'clf__max_depth': [3, 4, 5, 6], 'clf__subsample': [0.6, 0.8, 1.0], 'clf__colsample_bytree': [0.6, 0.8, 1.0]}
    ),
    'SVM': (
        Pipeline([
            ('prep', linear_preprocessor),           
            ('clf', CalibratedClassifierCV(
                estimator=LinearSVC(
                    dual=False,
                    max_iter=5000, 
                    class_weight='balanced',
                    random_state=42
                ),
                method='sigmoid',
                n_jobs=1
            ))
        ]),
        
        # Hyperparameters
        {
            'clf__estimator__C': [0.01, 0.1, 1, 10] 
        }
    )
}

### Train Model

In [33]:
import os
import time
import pandas as pd
import numpy as np
import pickle
import warnings
from sklearn.model_selection import RandomizedSearchCV, train_test_split, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

In [1]:
random_seeds = [42, 12, 15]
random_seeds_retry = [42, 12, 15]
fractions = [1] 
n_iter_search = 128

results_list = []
confusion_matrices = {}

print(f"   Starting Experiment...")
print(f"   Seeds: {random_seeds}")
print(f"   Fractions: {fractions}")
print(f"   Search Iterations: {n_iter_search}")
print("-" * 60)

   Starting Experiment...
   Seeds: [42, 12, 15]
   Fractions: [1]
   Search Iterations: 128
------------------------------------------------------------


In [ ]:
csv_save_path = '../results/final_experiment_best_model_results.csv'
pkl_save_path = '../results/final_experiment_best_model_results.pkl'

for seed in random_seeds_retry:
    print(f"\nRandom State: {seed}")
    
    # --- Step A: Global Split (60% Train, 20% Val, 20% Test) ---
    X_temp, X_test_seed, y_temp, y_test_seed = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    X_train_full_seed, X_val_seed, y_train_full_seed, y_val_seed = train_test_split(
        X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=seed
    )
    
    print(f"   Data Split -> Train: {X_train_full_seed.shape[0]}, Val: {X_val_seed.shape[0]}, Test: {X_test_seed.shape[0]}")

    for frac in fractions:
        print(f"\n   >>> Processing Fraction: {frac*100}%")
        
        # --- Step B: Subsample Train ---
        if frac == 1.0:
            X_train_sub, y_train_sub = X_train_full_seed, y_train_full_seed
        else:
            X_train_sub, _, y_train_sub, _ = train_test_split(
                X_train_full_seed, y_train_full_seed, 
                train_size=frac, stratify=y_train_full_seed, random_state=seed
            )
        
        n_train = X_train_sub.shape[0]
        
        # --- Step C: PredefinedSplit (Train + Val) ---
        X_combined = pd.concat([X_train_sub, X_val_seed], axis=0)
        y_combined = pd.concat([y_train_sub, y_val_seed], axis=0)
        
        test_fold = np.concatenate([
            np.full(X_train_sub.shape[0], -1),
            np.zeros(X_val_seed.shape[0])
        ])
        ps = PredefinedSplit(test_fold)
        
        # --- Step D: Model Loop ---
        for name, (pipeline, param_grid) in model_configs.items():

            print(f"      > {name}...", end=" ")
            start_time = time.time()

            if name == 'Logistic Regresion':
                n_iter_search = 6
            elif name == 'SVM':
                n_iter_search = 4
            elif name == 'Decision Tree':
                n_iter_search = 9
            elif name == 'Random Forest':
                n_iter_search = 48
            else:
                n_iter_search = 128

            try:
                # 1. Tuning
                search = RandomizedSearchCV(
                    pipeline, param_grid,
                    n_iter=n_iter_search,
                    scoring='average_precision', 
                    cv=ps, 
                    n_jobs=1, # -1
                    random_state=seed,
                    refit=True, 
                    verbose=0
                )
                
                search.fit(X_combined, y_combined)
                
                best_model = search.best_estimator_
                
                ### XGBoost using early stopping
                if name == 'XGBoost':
                    print("      > [XGBoost] Retraining with Early Stopping...", end=" ")
                    
                    prep_step = best_model.named_steps['prep']
                    
                    X_train_processed = prep_step.transform(X_train_sub)
                    X_val_processed = prep_step.transform(X_val_seed)
                    
                    raw_params = search.best_params_
                    xgb_params = {k.replace('clf__', ''): v for k, v in raw_params.items()}

                    xgb_params['n_estimators'] = 2000 
                    xgb_params['early_stopping_rounds'] = 50
                    xgb_params['eval_metric'] = 'aucpr'
                    xgb_params['random_state'] = seed
                    xgb_params['n_jobs'] = 1 # -1
                    

                    new_xgb = XGBClassifier(**xgb_params)
                    
                    new_xgb.fit(
                        X_train_processed, y_train_sub,
                        eval_set=[(X_val_processed, y_val_seed)],
                        verbose=False
                    )
                    
                    best_model = Pipeline([
                        ('prep', prep_step),
                        ('clf', new_xgb)
                    ])
                    
                    print(f"Done! Stopped at iter {new_xgb.best_iteration}")

            # --- Step E: Final Evaluation on Hold-out Test ---

                y_pred_proba = best_model.predict_proba(X_test_seed)[:, 1]
                y_pred_class = best_model.predict(X_test_seed)
                
                test_pr_auc = average_precision_score(y_test_seed, y_pred_proba)
                test_roc_auc = roc_auc_score(y_test_seed, y_pred_proba)
                best_val_score = search.best_score_

                # # 3. Save Confusion Matrix
                # cm = confusion_matrix(y_test_seed, y_pred_class)
                # cm_key = f"{name}_Seed{seed}_Frac{int(frac*100)}%"
                # confusion_matrices[cm_key] = cm
                
                elapsed = time.time() - start_time

                print(f"Done! ({elapsed:.1f}s)")
                print(f"        Best Params: {search.best_params_}")
                print(f"        Scores     : Val PR={best_val_score:.4f} | Test PR={test_pr_auc:.4f} | Test ROC={test_roc_auc:.4f}")
                print("-" * 40)
                
                # 4. Append Results
                results_list.append({
                    'Random_State': seed,
                    'Fraction': frac,
                    'Samples': n_train,
                    'Model': name,
                    'Best_Params': str(search.best_params_),
                    'Val_PR_AUC': best_val_score,
                    'Test_PR_AUC': test_pr_auc,
                    'Test_ROC_AUC': test_roc_auc,
                    'Time_Sec': elapsed
                })
                
                # Checkpoint Save
                pd.DataFrame(results_list).to_csv(csv_save_path, index=False)
                # with open(pkl_save_path, 'wb') as f:
                #     pickle.dump(confusion_matrices, f)
                    
            except Exception as e:
                print(f"\nError: {e}")
                pd.DataFrame(results_list).to_csv(csv_save_path, index=False)

print("\nFull Experiment Completed!")


Random State: 42
   Data Split -> Train: 184506, Val: 61502, Test: 61503

   >>> Processing Fraction: 100%
      > XGBoost...       > [XGBoost] Retraining with Early Stopping... Done! Stopped at iter 433
Done! (1480.3s)
        Best Params: {'clf__subsample': 0.6, 'clf__n_estimators': 800, 'clf__max_depth': 3, 'clf__learning_rate': 0.1, 'clf__colsample_bytree': 0.8}
        Scores     : Val PR=0.2441 | Test PR=0.2561 | Test ROC=0.7657
----------------------------------------

Random State: 12
   Data Split -> Train: 184506, Val: 61502, Test: 61503

   >>> Processing Fraction: 100%
      > XGBoost...       > [XGBoost] Retraining with Early Stopping... Done! Stopped at iter 582
Done! (1578.2s)
        Best Params: {'clf__subsample': 0.8, 'clf__n_estimators': 500, 'clf__max_depth': 4, 'clf__learning_rate': 0.05, 'clf__colsample_bytree': 0.8}
        Scores     : Val PR=0.2479 | Test PR=0.2545 | Test ROC=0.7673
----------------------------------------

Random State: 15
   Data Split -> Tr